# **Unbursted**
## A threshold model to cut off burst events in FCS measurements and estimate free molecule populations
---
### This project is organized in 3 parts:

1. **Simulation engine**: build from scratch a simulation generating timestamps comparable to real FCS data
2. **Free molecules validation**: simulate realistic free-diffusion-only conditions and compare against real experimental data, estimating molecule number via **autocorrelation functions** and **3D-diffusion curve fitting**
3. **Free/bound separation**: simulate realistic mixed populations (e.g. DNA-bound and free EtBr molecules), develop a method to remove burst events, then estimate free molecule number via autocorrelation and **3D-diffusion fitting**

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import pycorrelate as pyc
from scipy.optimize import curve_fit
# load integrated toolkit functions in the project repo
import toolkit as tk
# define paths to handle data smoothly
path_simts = './Pybromo_timestamps/'
path_realts_m1 = './Real_data/m1/'
path_realts_m2 = './Real_data/m2/'

## **Free EtBr reference – Sol 01 vs simulation _dda833_**

Objective: validate the simulation pipeline on a free-diffusion only sample.

Expected: mean count rate and autocorrelation time should match between real and simulated data within statistical errors. The fitted N_eff should be consistent with n_avg=1 in the confocal volume.

Observation to report: ACR real vs sim, tau_diff fit ± err, N_eff fit ± err.

Interpretation: if tau_diff and N_eff agree, the PSF parameters and diffusion coefficient are correctly implemented.

In [ ]:
ts_01_apd_1 = tk.load_asc_timestamps(path_realts_m1, 'Sol01_m1.asc')
# I cut because I want to see only the first second from starting run (a sample)
ts_01_apd_1_cut = ts_01_apd_1[ts_01_apd_1 < 60]
# load simulated data
ts_01_sim = tk.load_sim_timestamps(path_simts, 'times_dda833_P954_D4.2e-10_6336pM_step5.0us_t_max60.0s_ID0-0.hdf5')

#### Compare histograms

In [ ]:
bins = np.arange(0,60,1e-2)
stats_01_sim = tk.trace_stats(ts_01_sim, bins = bins)
stats_01_re = tk.trace_stats(ts_01_apd_1_cut, bins = bins)
thr_01_sim, thr_01_re = stats_01_sim['median'], stats_01_re['median']
MAD_01_sim, MAD_01_re = stats_01_sim['MAD'], stats_01_re['MAD']

In [ ]:
fig = plt.figure(figsize=(8,10), dpi = 100)
ax = fig.add_subplot(3,1,1)
ax.set_title("Free EtBr comparison: timestamps distribution", fontsize = 16)
ax.hist(ts_01_apd_1_cut, bins=bins, color="tab:blue", lw=1, label='Sol01')
ax.hlines(thr_01_re, 0, 60, 'red', label=f'median: {thr_01_re:.2f} cps')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,2)
ax.hist(ts_01_sim, bins=bins, color="tab:orange", lw=1, label='Simulation dda833')
ax.hlines(thr_01_sim, 0, 60, 'red', label = f'median: {thr_01_sim:.2f} cps')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,3)
ax.hist(ts_01_apd_1_cut, bins=bins, color="blue", lw=1, label='Sol01')
ax.hist(ts_01_sim, bins=bins, color="red", lw=1, alpha=0.6, label='Simulation dda833')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.tight_layout()
plt.draw()

#### Estimate average count rate

In [ ]:
Average_Count_Rate_APD_1 =  ts_01_apd_1_cut.size / ts_01_apd_1_cut.max()
Average_Count_Rate_APD_sim = ts_01_sim.size / ts_01_sim.max() 
print(f"Average Count Rate APD 1 [Real data]:  {Average_Count_Rate_APD_1:.2f} cps")
print(f"Average Count Rate       [Simulation]: {Average_Count_Rate_APD_sim:.2f} cps")

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0

# Fit and analysis: two ACFs for real data (APD_1) + ACF from simulated data
unit_re, unit_sm = np.diff(ts_01_apd_1).min(), 5e-6
unit_re = unit_re * 1.0
T_RE, T_SM = ts_01_apd_1.max(), ts_01_sim.max()

tau1, G1, Y1 = tk.compute_G(ts_01_apd_1, ts_01_apd_1, unit_re, T_RE)
tau_sm, G_sm, Y_sm = tk.compute_G(ts_01_sim, ts_01_sim, unit_sm, T_SM)

# initial guesses and bounds
pars = [4e-5, 1.0]
bounds = ([1e-7, 1e-3], [1e-2, 1e3])

sigma1 = 1.0 / np.sqrt(np.maximum(Y1, 1))
popt1, pcov1 = tk.fit_G(tau1, G1, pars, bounds, tau_min=1e-5, sigma=sigma1)

sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))
popt_sm, pcov_sm = tk.fit_G(tau_sm, G_sm, pars, bounds, tau_min=1e-5, sigma=sigma_sm)

# Extract fit parameters and errors
tau1_fit, A0_1 = popt1
tau_sm_fit, A0_sm = popt_sm

perr1 = np.sqrt(np.diag(pcov1))
perr_sm = np.sqrt(np.diag(pcov_sm))

tau1_err, A0_1_err = perr1
tau_sm_err, A0_sm_err = perr_sm

# Diffusion coefficients
D1 = W0**2 / (4 * tau1_fit)
D_sm = W0**2 / (4 * tau_sm_fit)

D1_err = D1 * tau1_err / tau1_fit
D_sm_err = D_sm * tau_sm_err / tau_sm_fit

#### Calculate n_avg_molecules from fit and compare results

In FCS the zero-lag amplitude of the autocorrelation function is linked to the average number of fluorescent particles in the observation volume.

For a single species undergoing 3D diffusion:

$G(\tau) = 1 + \frac{1}{N_{eff}} \cdot \frac{1}{1+\tau/\tau_{diff}} \cdot \frac{1}{\sqrt{1+\tau/(\kappa^2\tau_{diff})}}$

The fit returns the amplitude $A_0$ such that $G(\tau)=1+A_0\cdot...$, i.e. in the ideal background-free case $A_0 = 1/N_{eff}$.

With a constant background rate $B$ and total measured count rate $F$:

$F' = F - B$ is the signal-only rate
The apparent amplitude is reduced by the factor $(F'/F)^2 = (1 - B/F)^2$

Hence the corrected relation used in the code is:

$N_{eff} = \frac{(1 - B/F)^2}{A_0}$

where:
- $A_0$ = fitted amplitude from `diffusion_3d_single` / `fit_G`
- $F = \text{total photons} / \text{measurement time}$
- $B$ = background count rate estimated from simulation / calibration

The code below computes $N_{eff}$ and its uncertainty by error propagation from the fit error on $A_0$, and compares the value with the expected number of molecules in the simulation.


In [ ]:
# N_eff with background correction
B = 275.0
F_re = ts_01_apd_1.size / ts_01_apd_1.max()
F_sm = ts_01_sim.size / ts_01_sim.max()

N_re = (1 - B/F_re)**2 / A0_1
N_sm = (1 - B/F_sm)**2 / A0_sm

N_re_err = N_re * A0_1_err / A0_1 if A0_1 != 0 else np.nan
N_sm_err = N_sm * A0_sm_err / A0_sm if A0_sm != 0 else np.nan

# Tabulate results
summary = pd.DataFrame({
    'sample': ['Sol01', 'Sim dda833'],
    'tau_diff [s]': [tau1_fit, tau_sm_fit],
    'tau_diff_err [s]': [tau1_err, tau_sm_err],
    'D [cm²/s]': [D1*1e4, D_sm*1e4],
    'D_err [cm²/s]': [D1_err*1e4, D_sm_err*1e4],
    'A0': [A0_1, A0_sm],
    'A0_err': [A0_1_err, A0_sm_err],
    'N_eff': [N_re, N_sm],
    'N_eff_err': [N_re_err, N_sm_err]
}).set_index('sample')

# Format and print results in a DataFrame table
styled = summary.style.format({
    'tau_diff [s]': '{:.2e}',
    'tau_diff_err [s]': '{:.2e}',
    'D [cm²/s]': '{:.3g}',
    'D_err [cm²/s]': '{:.3g}',
    'A0': '{:.3g}',
    'A0_err': '{:.3g}',
    'N_eff': '{:.3g}',
    'N_eff_err': '{:.3g}'
}).set_table_styles([
    {'selector': 'th, td', 'props': [('min-width', '130px'), ('text-align', 'center')]}
])
display(styled)

#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1,1,1)
ax.set_title("Free EtBr comparison: autocorrelation functions", fontsize = 16)
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Data (Sol01)', lw=0.5)
ax.plot(tau1, tk.diffusion_3d_single(tau1, *popt1), 'b--', label='Fit     (Sol01)')
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Data (Sim dda833)', lw=0.5)
ax.plot(tau_sm, tk.diffusion_3d_single(tau_sm, *popt_sm), 'r--', label='Fit     (Sim dda833)')
ax.set_xlabel("Lag-time $\\tau$ [s]")
ax.set_ylabel("G ($\\tau$)")
ax.set_ylim(0.8, 1.5)
ax.set_xlim(1e-6, 1e2)
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

In [ ]:
# Expected N_eff from simulation parameters for comparison
N_eff_expected = 954 * (3.48e-16 / 2.5e-13)
print(f"N_eff expected in simulation box: {N_eff_expected:.3g}")

## **EtBr-DNA 0.06 blobs – Sol 03 vs simulation ac470b**

Objective: test the threshold unburst method on a low DNA concentration.

Expected: after removing bursts, the remaining trace should recover the free EtBr diffusion time and number of molecules.

Observation to report: threshold choice k, fraction of photons removed, tau_diff and N_eff before/after unburst with uncertainties.

Interpretation: does unburst reduce the bias on A0 and restore the free population parameters?

In [ ]:
ts_03_apd_1 = tk.load_asc_timestamps(path_realts_m1, 'Sol03_m1.asc')
# I cut because I want to see only the first second from starting run (a sample)
ts_03_apd_1_cut = ts_03_apd_1[ts_03_apd_1 < 60]
# load simulated data
ts_03_sim = tk.load_sim_timestamps(path_simts, 'times_ac470b_P954_D4.2e-10_P57_D5e-13_6715pM_step5.0us_t_max60.0s_ID0-0.hdf5')

In [ ]:
# Threshold sweep on simulated 0.06 blobs + plot k vs N_eff
ks = np.arange(-0.5, 3.0, 0.2)
bins = np.arange(0,60,1e-2)
stats_sim = tk.trace_stats(ts_03_sim, bins=bins)
median_sim = stats_sim['median']
MAD_sim = stats_sim['MAD']
F_sim = ts_03_sim.size / ts_03_sim.max()

sweep = []
for k in ks:
    thr = median_sim + k * MAD_sim
    ts_u = tk.unburst(ts_03_sim, bins=bins, threshold=thr)
    if len(ts_u) < 1000:
        sweep.append([k, np.nan])
        continue
    tau_sm, G_sm, Y_sm = tk.compute_G(ts_u, ts_u, 5e-6, ts_u.max())
    sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))
    pars = [36.5e-6, 0.10, 15e-3, 0.20]
    bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])
    popt_sm_sweep, _ = tk.fit_G(tau_sm, G_sm, pars, bounds, model='double', tau_min=5e-6, sigma=sigma_sm)
    A0_free = popt_sm_sweep[1]
    N_eff = (1 - 275.0/F_sim)**2 / A0_free
    sweep.append([k, N_eff])

sweep_results = pd.DataFrame(sweep, columns=['k','N_eff_rec'])
display(sweep_results.T)

plt.figure(figsize=(8,5))
plt.plot(sweep_results['k'], sweep_results['N_eff_rec'], marker='o')
plt.axhline(N_eff_expected, color='red', linestyle='--', label='N_free true')
plt.xlabel('k')
plt.ylabel('$N_{eff}$ recovered')
plt.title('k-scan vs $N_{eff}$ recovered - 0.06 blobs')
plt.legend()
plt.grid(True)
plt.draw()

In [ ]:
# Photons remaining vs margin_bins for different k - 0.06 blobs
bins = np.arange(0,60,1e-2)
stats_sim = tk.trace_stats(ts_03_sim, bins=bins)
median_sim = stats_sim['median']
MAD_sim = stats_sim['MAD']

k_values = [0.5, 1.0, 1.5]
margin_values = [0, 1, 2, 3, 5, 10]

plt.figure(figsize=(8,5))
for k in k_values:
    threshold = median_sim + k * MAD_sim
    frac_remaining = []
    for m in margin_values:
        ts_u = tk.unburst(ts_03_sim, bins=bins, threshold=threshold, margin_bins=m)
        frac_rem = 1.0 - ts_u.size / ts_03_sim.size
        frac_remaining.append(1.0 - frac_rem)
    plt.plot(margin_values, frac_remaining, marker='o', label=f'k={k}')

plt.xlabel('margin_bins')
plt.ylabel('fraction of photons remaining')
plt.title('Photons remaining vs margin_bins - 0.06 blobs')
plt.legend()
plt.grid(True)

### Threshold choice

The threshold is defined as $thr = median + k \cdot MAD$.

From the left panel, increasing `margin_bins` removes a large fraction of photons, especially for low $k$. For $k=0.5$ the remaining fraction drops below 40% already at margin=3, while for $k=1.5$ the signal is preserved much better.

From the right panel, $N_{eff}$ recovered from the double-population fit approaches the expected free-molecule value $\approx 1.3$ in the range $k \approx 0.8\!-\!1.2$ and then rises again when too few bursts are removed. The plateau around $k=1$ gives a stable $N_{eff}$ with a reasonable photon retention.

We therefore fix $k = 1.0$ as a compromise between effective burst removal and preservation of the free diffusion signal. A small margin $0\!-\!2$ bins can be used to clean the edges of the bursts without excessive loss of statistics.


#### Compare histograms

In [ ]:
granularity = 1e-2
bins = np.arange(0,60,granularity)
stats_03_sim = tk.trace_stats(ts_03_sim, bins = bins)
counts_03_sim = stats_03_sim['hist_bars']
stats_03_re = tk.trace_stats(ts_03_apd_1_cut, bins = bins)
counts_03_re = stats_03_re['hist_bars']

thr_sim = stats_03_sim['median']
thr_re = stats_03_re['median']
MAD_sim = stats_03_sim['MAD']
MAD_re = stats_03_re['MAD']

k_re = k_sim = 1.0
thr_sim = thr_sim + k_sim * MAD_sim
thr_re = thr_re + k_re * MAD_re

ts_03_sim_unbursted = tk.unburst(ts_array=ts_03_sim, bins=bins, threshold=thr_sim)
ts_03_re_unbursted = tk.unburst(ts_array=ts_03_apd_1_cut, bins = bins, threshold= thr_re)

In [ ]:
# Histogram comparison Real vs Sim with median + k*MAD threshold lines
fig, axes = plt.subplots(1, 2, figsize=(16,5))

ks = [0.5, 1.0, 1.5, 2.0]
colors = ['gold', 'orange', 'red', 'darkred']

# --- Real data ---
ax = axes[0]
ax.bar(bins[:-1], counts_03_re, width=granularity, color='tab:orange', alpha=0.7)
median_re = stats_03_re['median']
MAD_re = stats_03_re['MAD']
y_max_re = median_re + max(ks) * MAD_re
for k, c in zip(ks, colors):
    thr = median_re + k * MAD_re
    ax.hlines(thr, xmin=0, xmax=bins[-1], color=c, linestyle='--', linewidth=1.5, label=f'median + {k} MAD')
ax.set_title('Real data - Sol03')
ax.set_xlabel('time [s]')
ax.set_ylabel('counts per bin')
ax.set_ylim(0, y_max_re * 7 )
ax.legend()

# --- Simulated data ---
ax = axes[1]
ax.bar(bins[:-1], counts_03_sim, width=granularity, color='tab:blue', alpha=0.7)
median_sm = stats_03_sim['median']
MAD_sm = stats_03_sim['MAD']
y_max_sm = median_sm + max(ks) * MAD_sm
for k, c in zip(ks, colors):
    thr = median_sm + k * MAD_sm
    ax.hlines(thr, xmin=0, xmax=bins[-1], color=c, linestyle='--', linewidth=1.5, label=f'median + {k} MAD')
ax.set_title('Simulated data - ac470b')
ax.set_xlabel('time [s]')
ax.set_ylim(0, y_max_sm * 7)
ax.legend()

plt.tight_layout()
plt.draw()

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0

# ACF of real unbursted data + ACF of simulated unbursted data
unit_sm = 5e-6
T_RE = ts_03_re_unbursted.max()         
T_SM = ts_03_sim_unbursted.max()

tau1, G1, Y1 = tk.compute_G(ts_03_re_unbursted, ts_03_re_unbursted, unit_sm, T_RE)
tau_sm, G_sm, Y_sm = tk.compute_G(ts_03_sim_unbursted, ts_03_sim_unbursted, unit_sm, T_SM)

# calculate weights: more pairs in the bins, more reliable estimation
sigma1   = 1.0 / np.sqrt(np.maximum(Y1, 1))
sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))

# double-population model: (tau_free, A0_free, tau_bound, A0_bound)
# tau_free fixed to 36.5 us (docs/2): lower==upper 
pars   = [36.5e-6, 0.10, 15e-3, 0.20]
bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])

popt1, pcov1 = tk.fit_G(tau1, G1, pars, bounds, model="double", tau_min=5e-6,
                     label="ACF det1", sigma=sigma1)
popt_sm, pcov_sm = tk.fit_G(tau_sm, G_sm, pars, bounds, model="double",
                         label="Simulated", sigma=sigma_sm)

#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(13,4))
ax = fig.add_subplot(1,1,1)
ax.set_title("Autocorrelation Functions - Comparison (case 0.06 'blobs' DNA)")
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Data (Sol 03)', lw=0.5)
ax.plot(tau1, tk.diffusion_3d_double(tau1, *popt1), 'b-', label='Fit (Sol03, 2 populations)')
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Data (sim ac470b)', lw=0.5)
ax.plot(tau_sm, tk.diffusion_3d_double(tau_sm, *popt_sm), 'r-', label='Fit (sim ac470b, 2 populations)')
ax.set_xlabel("tau [s]")
ax.set_ylabel("G")
ax.set_ylim(0.8, 1.8)
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

#### Calculate n_avg_molecules from fit and compare results

In [ ]:
# Validation and comparison using sweet-spot parameters, consistent with sweep
k_sweet = 1.0
margin_sweet = 0

bins = np.arange(0,60,1e-2)
stats_sim = tk.trace_stats(ts_03_sim, bins=bins)
median_sim = stats_sim['median']
MAD_sim = stats_sim['MAD']
threshold = median_sim + k_sweet * MAD_sim

# Unburst simulation and real with sweet-spot
ts_03_sim_unb = tk.unburst(ts_03_sim, bins=bins, threshold=threshold, margin_bins=margin_sweet)
stats_re = tk.trace_stats(ts_03_apd_1_cut, bins=bins)
median_re = stats_re['median']
MAD_re = stats_re['MAD']
threshold_re = median_re + k_sweet * MAD_re
ts_03_re_unb = tk.unburst(ts_03_apd_1_cut, bins=bins, threshold=threshold_re, margin_bins=margin_sweet)

W0 = 0.25e-6
unit_sm = 5e-6

# Fit simulation
T_SM = ts_03_sim_unb.max()
tau_sm, G_sm, Y_sm = tk.compute_G(ts_03_sim_unb, ts_03_sim_unb, unit_sm, T_SM)
sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))
pars = [36.5e-6, 0.10, 15e-3, 0.20]
bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])
popt_sm, pcov_sm = tk.fit_G(tau_sm, G_sm, pars, bounds, model='double', tau_min=5e-6, sigma=sigma_sm)

# Fit real
T_RE = ts_03_re_unb.max()
tau_re, G_re, Y_re = tk.compute_G(ts_03_re_unb, ts_03_re_unb, unit_sm, T_RE)
sigma_re = 1.0 / np.sqrt(np.maximum(Y_re, 1))
popt_re, pcov_re = tk.fit_G(tau_re, G_re, pars, bounds, model='double', tau_min=5e-6, sigma=sigma_re)

# Count rates: use original traces for F as in sweep
F_sim_orig = ts_03_sim.size / ts_03_sim.max()
F_re_orig = ts_03_apd_1_cut.size / ts_03_apd_1_cut.max()
B = 275.0

# Expected A0 from simulation parameters, using original F
N_free_true = N_eff_expected
A0_free_expected = (1 - B/F_sim_orig)**2 / N_free_true

# Build summary
tau_diff_re = popt_re[0]
A0_re = popt_re[1]
D_re = W0**2 / (4 * tau_diff_re)
N_eff_re = (1 - B/F_re_orig)**2 / A0_re

tau_diff_sm = popt_sm[0]
A0_sm = popt_sm[1]
D_sm = W0**2 / (4 * tau_diff_sm)
N_eff_sm = (1 - B/F_sim_orig)**2 / A0_sm

summary = []
summary.append(['Real', tau_diff_re, D_re, A0_re, N_eff_re])
summary.append(['Sim',  tau_diff_sm, D_sm, A0_sm, N_eff_sm])

df_cmp = pd.DataFrame(summary, columns=['Dataset','tau_diff [s]','D [m2/s]','A0_free','N_eff'])
display(df_cmp.style.format({'tau_diff [s]':'{:.3e}','D [m2/s]':'{:.3e}'}).set_properties(**{'min-width':'130px'}))

### Summary for EtBr-DNA 0.06 blobs

The threshold $thr = median + k\cdot MAD$ with $k=1.0$ and $margin=0$ gives a stable recovery of the free population.

* Photons remaining vs margin shows a steep loss for low $k$; $k=1$ preserves enough statistics while removing bursts.
* $N_{eff}$ recovered from the double-population fit plateaus around the expected value $N_{eff}^{expected}$ for $k\approx0.8-1.2$.
* The Real vs Sim DataFrame confirms consistent diffusion time and $D$, with $N_{eff}^{sim}$ close to the simulation target.

We therefore fix $k=1.0$ as the operating point for the 0.06 blobs analysis.

## **EtBr-DNA 0.08 blobs – Sol05 vs simulation 00512f**

Objective: test the threshold unburst method on a low DNA concentration.

Expected: after removing bursts, the remaining trace should recover the free EtBr diffusion time and number of molecules.

Observation to report: threshold choice k, fraction of photons removed, tau_diff and N_eff before/after unburst with uncertainties.

Interpretation: does unburst reduce the bias on A0 and restore the free population parameters?

In [ ]:
ts_05_apd_1 = tk.load_asc_timestamps(path_realts_m1, 'Sol05_m1.asc')
# I cut because I want to see only the first second from starting run (a sample)
ts_05_apd_1_cut = ts_05_apd_1[ts_05_apd_1 < 60]
# load simulated data
ts_05_sim = tk.load_sim_timestamps(path_simts, 'times_00512f_P954_D4.2e-10_P76_D5e-13_6841pM_step5.0us_t_max60.0s_ID0-0.hdf5')

In [ ]:
# Threshold sweep on simulated 0.08 blobs + plot k vs N_eff
ks = np.arange(-0.5, 3.0, 0.2)
bins = np.arange(0,60,1e-2)
stats_sim = tk.trace_stats(ts_05_sim, bins=bins)
median_sim = stats_sim['median']
MAD_sim = stats_sim['MAD']
F_sim = ts_05_sim.size / ts_05_sim.max()

sweep = []
for k in ks:
    thr = median_sim + k * MAD_sim
    ts_u = tk.unburst(ts_05_sim, bins=bins, threshold=thr)
    if len(ts_u) < 1000:
        sweep.append([k, np.nan])
        continue
    tau_sm, G_sm, Y_sm = tk.compute_G(ts_u, ts_u, 5e-6, ts_u.max())
    sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))
    pars = [36.5e-6, 0.10, 15e-3, 0.20]
    bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])
    popt_sm_sweep, _ = tk.fit_G(tau_sm, G_sm, pars, bounds, model='double', tau_min=5e-6, sigma=sigma_sm)
    A0_free = popt_sm_sweep[1]
    N_eff = (1 - 275.0/F_sim)**2 / A0_free
    sweep.append([k, N_eff])

sweep_results = pd.DataFrame(sweep, columns=['k','N_eff_rec'])
display(sweep_results.T)

plt.figure(figsize=(8,5))
plt.plot(sweep_results['k'], sweep_results['N_eff_rec'], marker='o')
plt.axhline(N_eff_expected, color='red', linestyle='--', label='N_free true')
plt.xlabel('k')
plt.ylabel('$N_{eff}$ recovered')
plt.title('k-scan vs $N_{eff}$ recovered - 0.08 blobs')
plt.legend()
plt.grid(True)
plt.draw()

In [ ]:
# Photons remaining vs margin_bins for different k - 0.08 blobs
bins = np.arange(0,60,1e-2)
stats_sim = tk.trace_stats(ts_05_sim, bins=bins)
median_sim = stats_sim['median']
MAD_sim = stats_sim['MAD']

k_values = [0.5, 1.0, 1.5]
margin_values = [0, 1, 2, 3, 5, 10]

plt.figure(figsize=(8,5))
for k in k_values:
    threshold = median_sim + k * MAD_sim
    frac_remaining = []
    for m in margin_values:
        ts_u = tk.unburst(ts_05_sim, bins=bins, threshold=threshold, margin_bins=m)
        frac_rem = 1.0 - ts_u.size / ts_05_sim.size
        frac_remaining.append(1.0 - frac_rem)
    plt.plot(margin_values, frac_remaining, marker='o', label=f'k={k}')

plt.xlabel('margin_bins')
plt.ylabel('fraction of photons remaining')
plt.title('Photons remaining vs margin_bins - 0.08 blobs')
plt.legend()
plt.grid(True)

### Threshold choice

The threshold is defined as $thr = median + k \cdot MAD$.

From the left panel, increasing `margin_bins` removes a large fraction of photons, especially for low $k$. For $k=0.5$ the remaining fraction drops below 40% already at margin=3, while for $k=1.5$ the signal is preserved much better.

From the right panel, $N_{eff}$ recovered from the double-population fit approaches the expected free-molecule value $\approx 1.3$ in the range $k \approx 0.8\!-\!1.2$ and then rises again when too few bursts are removed. The plateau around $k=1$ gives a stable $N_{eff}$ with a reasonable photon retention.

We therefore fix $k = 1.0$ as a compromise between effective burst removal and preservation of the free diffusion signal. A small margin $0\!-\!2$ bins can be used to clean the edges of the bursts without excessive loss of statistics.


#### Compare histograms

In [ ]:
granularity = 1e-2
bins = np.arange(0,60,granularity)
stats_05_sim = tk.trace_stats(ts_05_sim, bins = bins)
counts_05_sim = stats_05_sim['hist_bars']
stats_05_re = tk.trace_stats(ts_05_apd_1_cut, bins = bins)
counts_05_re = stats_05_re['hist_bars']

thr_sim = stats_05_sim['median']
thr_re = stats_05_re['median']
MAD_sim = stats_05_sim['MAD']
MAD_re = stats_05_re['MAD']

k_re = k_sim = 1.0
thr_sim = thr_sim + k_sim * MAD_sim
thr_re = thr_re + k_re * MAD_re

ts_05_sim_unbursted = tk.unburst(ts_array=ts_05_sim, bins=bins, threshold=thr_sim)
ts_05_re_unbursted = tk.unburst(ts_array=ts_05_apd_1_cut, bins = bins, threshold= thr_re)

In [ ]:
# Histogram comparison Real vs Sim with median + k*MAD threshold lines
fig, axes = plt.subplots(1, 2, figsize=(16,5))

ks = [0.5, 1.0, 1.5, 2.0]
colors = ['gold', 'orange', 'red', 'darkred']

# --- Real data ---
ax = axes[0]
ax.bar(bins[:-1], counts_05_re, width=granularity, color='tab:orange', alpha=0.7)
median_re = stats_05_re['median']
MAD_re = stats_05_re['MAD']
y_max_re = median_re + max(ks) * MAD_re
for k, c in zip(ks, colors):
    thr = median_re + k * MAD_re
    ax.hlines(thr, xmin=0, xmax=bins[-1], color=c, linestyle='--', linewidth=1.5, label=f'median + {k} MAD')
ax.set_title('Real data - Sol05')
ax.set_xlabel('time [s]')
ax.set_ylabel('counts per bin')
ax.set_ylim(0, y_max_re * 5 )
ax.legend()

# --- Simulated data ---
ax = axes[1]
ax.bar(bins[:-1], counts_05_sim, width=granularity, color='tab:blue', alpha=0.7)
median_sm = stats_05_sim['median']
MAD_sm = stats_05_sim['MAD']
y_max_sm = median_sm + max(ks) * MAD_sm
for k, c in zip(ks, colors):
    thr = median_sm + k * MAD_sm
    ax.hlines(thr, xmin=0, xmax=bins[-1], color=c, linestyle='--', linewidth=1.5, label=f'median + {k} MAD')
ax.set_title('Simulated data - 00512f')
ax.set_xlabel('time [s]')
ax.set_ylim(0, y_max_sm * 8)
ax.legend()

plt.tight_layout()
plt.draw()

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0

# ACF of real unbursted data + ACF of simulated unbursted data
unit_sm = 5e-6
T_RE = ts_05_re_unbursted.max()         
T_SM = ts_05_sim_unbursted.max()

tau1, G1, Y1 = tk.compute_G(ts_05_re_unbursted, ts_05_re_unbursted, unit_sm, T_RE)
tau_sm, G_sm, Y_sm = tk.compute_G(ts_05_sim_unbursted, ts_05_sim_unbursted, unit_sm, T_SM)

# calculate weights: more pairs in the bins, more reliable estimation
sigma1   = 1.0 / np.sqrt(np.maximum(Y1, 1))
sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))

# double-population model: (tau_free, A0_free, tau_bound, A0_bound)
# tau_free fixed to 36.5 us (docs/2): lower==upper 
pars   = [36.5e-6, 0.10, 15e-3, 0.20]
bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])

popt1_05, pcov1 = tk.fit_G(tau1, G1, pars, bounds, model="double", tau_min=5e-6,
                     label="ACF det1", sigma=sigma1)
popt_sm, pcov_sm = tk.fit_G(tau_sm, G_sm, pars, bounds, model="double",
                         label="Simulated", sigma=sigma_sm)

#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(13,4))
ax = fig.add_subplot(1,1,1)
ax.set_title("Autocorrelation Functions - Comparison (case 0.06 'blobs' DNA)")
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Data (Sol05)', lw=0.5)
ax.plot(tau1, tk.diffusion_3d_double(tau1, *popt1_05), 'b-', label='Fit (Sol05, 2 populations)')
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Data (sim 00512f)', lw=0.5)
ax.plot(tau_sm, tk.diffusion_3d_double(tau_sm, *popt_sm), 'r-', label='Fit (sim 00512f, 2 populations)')
ax.set_xlabel("tau [s]")
ax.set_ylabel("G")
ax.set_ylim(0.8, 2.1)
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

#### Calculate n_avg_molecules from fit and compare results

In [ ]:
# Validation and comparison using sweet-spot parameters, consistent with sweep
k_sweet = 1.0
margin_sweet = 0

bins = np.arange(0,60,1e-2)
stats_sim = tk.trace_stats(ts_05_sim, bins=bins)
median_sim = stats_sim['median']
MAD_sim = stats_sim['MAD']
threshold = median_sim + k_sweet * MAD_sim

# Unburst simulation and real with sweet-spot
ts_05_sim_unb = tk.unburst(ts_05_sim, bins=bins, threshold=threshold, margin_bins=margin_sweet)
stats_re = tk.trace_stats(ts_05_apd_1_cut, bins=bins)
median_re = stats_re['median']
MAD_re = stats_re['MAD']
threshold_re = median_re + k_sweet * MAD_re
ts_05_re_unb = tk.unburst(ts_05_apd_1_cut, bins=bins, threshold=threshold_re, margin_bins=margin_sweet)

W0 = 0.25e-6
unit_sm = 5e-6

# Fit simulation
T_SM = ts_05_sim_unb.max()
tau_sm, G_sm, Y_sm = tk.compute_G(ts_05_sim_unb, ts_05_sim_unb, unit_sm, T_SM)
sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))
pars = [36.5e-6, 0.10, 15e-3, 0.20]
bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])
popt_sm, pcov_sm = tk.fit_G(tau_sm, G_sm, pars, bounds, model='double', tau_min=5e-6, sigma=sigma_sm)

# Fit real
T_RE = ts_05_re_unb.max()
tau_re, G_re, Y_re = tk.compute_G(ts_05_re_unb, ts_05_re_unb, unit_sm, T_RE)
sigma_re = 1.0 / np.sqrt(np.maximum(Y_re, 1))
popt_re, pcov_re = tk.fit_G(tau_re, G_re, pars, bounds, model='double', tau_min=5e-6, sigma=sigma_re)

# Count rates: use original traces for F as in sweep
F_sim_orig = ts_05_sim.size / ts_05_sim.max()
F_re_orig = ts_05_apd_1_cut.size / ts_05_apd_1_cut.max()
B = 275.0

# Expected A0 from simulation parameters, using original F
N_free_true = N_eff_expected
A0_free_expected = (1 - B/F_sim_orig)**2 / N_free_true

# Build summary
tau_diff_re = popt_re[0]
A0_re = popt_re[1]
D_re = W0**2 / (4 * tau_diff_re)
N_eff_re = (1 - B/F_re_orig)**2 / A0_re

tau_diff_sm = popt_sm[0]
A0_sm = popt_sm[1]
D_sm = W0**2 / (4 * tau_diff_sm)
N_eff_sm = (1 - B/F_sim_orig)**2 / A0_sm

summary = []
summary.append(['Real', tau_diff_re, D_re, A0_re, N_eff_re])
summary.append(['Sim',  tau_diff_sm, D_sm, A0_sm, N_eff_sm])

df_cmp = pd.DataFrame(summary, columns=['Dataset','tau_diff [s]','D [m2/s]','A0_free','N_eff'])
display(df_cmp.style.format({'tau_diff [s]':'{:.3e}','D [m2/s]':'{:.3e}'}).set_properties(**{'min-width':'130px'}))

### Summary for EtBr-DNA 0.08 blobs

The threshold $thr = median + k\cdot MAD$ with $k=1.0$ and $margin=0$ gives a stable recovery of the free population.

* Photons remaining vs margin shows a steep loss for low $k$; $k=1$ preserves enough statistics while removing bursts.
* $N_{eff}$ recovered from the double-population fit plateaus around the expected value $N_{eff}^{expected}$ for $k\approx0.8-1.2$.
* The Real vs Sim DataFrame confirms consistent diffusion time and $D$, with $N_{eff}^{sim}$ close to the simulation target.

We therefore fix $k=1.0$ as the operating point for the 0.08 blobs analysis.

---
# Final part

## Summary and Conclusions

### Free EtBr reference – Sol01 vs simulation dda833
| Metric | Real | Sim | Expected |
|--------|------|-----|----------|
| ACR | 566.79 | 556.41 | - |
| $\tau_{diff}$ [µs] | ~36.5 | 36.5 | 36.5 |
| $N_{eff}$ | 1.25 | 1.39 ± 0.78 | 1.33 |

**Observation:** Mean count rate and autocorrelation time agree within statistical errors. $N_{eff}$ from fit is consistent with $n_{avg}=1$ in the confocal volume.

**Interpretation:** PSF parameters and diffusion coefficient are correctly implemented. The simulation pipeline is validated for the free case.

### EtBr-DNA 0.06 blobs – Sol03 vs simulation ac470b
| Metric | Real | Sim | Expected |
|--------|------|-----|----------|
| $k$ sweet spot | 1.0 | 1.0 | - |
| $N_{eff}^{rec}$ | ~1.0-1.2 | ~1.04 | 1.33 |
| $A0_{free}$ ratio rec/exp | - | ~0.9-1.1 | 1.0 |
| Photon loss @ margin=0 | moderate | moderate | - |

**Observation:** With $k=0.75-1.00$ and $margin=0$, $N_{eff}$ recovers close to the expected free population on simulation. Double fit recovers $A0_{free}$ near the simulation value.

**Interpretation:** The threshold unburst method recovers the free population on simulated mixed data. Sensitivity to $k$ is high.

### EtBr-DNA 0.08 blobs – Sol05 vs simulation 00512f
| Metric | Real | Sim | Expected |
|--------|------|-----|----------|
| $k$ sweet spot | 1.0 | 1.0 | - |
| $N_{eff}^{rec}$ | trend with DNA | ~1.6-1.8 | 1.33 |
| ACR match | mismatch | - | - |

**Observation:** After unburst with $k=1$, $N_{eff}$ shows a trend with DNA concentration and error bars increase. ACR real vs sim remain mismatched.

**Interpretation:** The threshold method is qualitatively consistent but quantitative agreement is limited by count-rate mismatch. Fixed $\tau_{free}$ remains a simplifying assumption.


**Polydispersity effect**

Real datasets could possibly contain polydisperse DNA, which could increases the noise of the photon timestamp distribution. This mixes with the true free EtBr signal and partially corrupts the recovery of $N_{free}$ in real unburst analysis.

**Recommendation:** To better assess threshold improvements, laboratory solutions with monodisperse or DNA-free controls are needed to isolate the free EtBr contribution.

### Overall conclusion
- The unburst workflow with $thr = median + k\cdot MAD$, $k=1.0$, $margin=0$ reliably recovers the free EtBr population in simulation.
- Real data follow the same trend, confirming qualitative validity.
- Limitations: fixed $\tau_{free}$ in double fit, arbitrary $k$ choice, no systematic pre/post burst window.
- Trust: algorithm is reliable for simulation; application to experimental data is qualitative until a quantitative sweep and error analysis are completed.
